# Strong-Strong Tracking Task

This notebook builds the same strong-strong tracking case as `examples/strong_strong_tracking.jl`, but all runtime choices are ordinary Julia variables. It uses no shell environment-variable overrides.

The configuration cell is set up for a long CUDA PIC run by default. For a quick smoke test, reduce `turns`, `n_macro_ele`, and `n_macro_pro`, or set `use_gpu = false`.


In [ ]:
import Pkg
project_root = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(project_root)

if !isdefined(Main, :Octopus)
    include(joinpath(project_root, "src", "Octopus.jl"))
end
using .Octopus

println("Project root: ", project_root)

In [ ]:
# Runtime configuration. Edit these values directly in the notebook.
turns = 50_000
n_macro_ele = 2_560_000
n_macro_pro = 1_024_000

use_gpu = true
cuda_device = nothing             # keep current CUDA device; use 0, 1, ... to select explicitly
solver_kind = :PIC                # :PIC or :gaussian
pic_green_cache = :slice_pair     # :none or :slice_pair
pic_longitudinal_kick = true
pic_batch_mode = :wavefront       # :sequential or :wavefront
pic_luminosity_every = 100        # 1 = every turn, 0 = disabled, N = every N turns

# CUDA PIC experimental/performance switches. These are set here from Julia
# variables instead of relying on shell environment overrides.
cuda_pic_indexed_wavefront = true
cuda_pic_timing = false           # turn on only for short profiling runs
cuda_pic_timing_detail = false
pic_cache_stats = false
cuda_memory_log_every = 0         # 0 disables periodic CUDA memory logging

function set_octopus_env!(name, enabled_or_value)
    if enabled_or_value === false || enabled_or_value === nothing || enabled_or_value == 0
        delete!(ENV, name)
    elseif enabled_or_value === true
        ENV[name] = "1"
    else
        ENV[name] = string(enabled_or_value)
    end
    return nothing
end

set_octopus_env!("OCTOPUS_CUDA_PIC_INDEXED_WAVEFRONT", cuda_pic_indexed_wavefront)
set_octopus_env!("OCTOPUS_CUDA_PIC_TIMING", cuda_pic_timing)
set_octopus_env!("OCTOPUS_CUDA_PIC_TIMING_DETAIL", cuda_pic_timing_detail)
set_octopus_env!("OCTOPUS_PIC_CACHE_STATS", pic_cache_stats)
set_octopus_env!("OCTOPUS_CUDA_MEMORY_LOG_EVERY", cuda_memory_log_every)

if use_gpu
    import CUDA
    CUDA.functional(false) || error("use_gpu=true requested, but CUDA.functional(false) is false.")
    policy = GPUExecutionPolicy(device = cuda_device)
else
    policy = CPUThreadsExecutionPolicy()
end

(
    policy = policy,
    turns = turns,
    n_macro_ele = n_macro_ele,
    n_macro_pro = n_macro_pro,
    solver_kind = solver_kind,
    pic_green_cache = pic_green_cache,
    pic_batch_mode = pic_batch_mode,
    pic_luminosity_every = pic_luminosity_every,
    cuda_pic_indexed_wavefront = get(ENV, "OCTOPUS_CUDA_PIC_INDEXED_WAVEFRONT", "0"),
)


In [ ]:
input = (
    case_name = "pic_hcc_notebook",
    result_dir = joinpath(project_root, "result"),
    seed = 123456789,
    total_turns = 50000,
    default_demo_macroparticles = 200,
    crossing_angle = 12.5e-3,

    electron = (
        charge = -1.0,
        mass = EMASS_EV,
        energy = 10.0e9,
        n_particle = 1.7203e11,
        n_macro = 2560000,
        cutoff = 5.0,
        sigma = (106.0e-6, 9.5e-6, 0.7e-2),
        beta = (0.55, 0.056, 0.7e-2 / 5.5e-4),
        alpha = (0.0, 0.0),
        alpha6 = (0.0, 0.0, 0.0),
        crab_beta = (150.0, 30.0, 0.7e-2 / 5.5e-4),
        tune = (0.08, 0.14, -0.069),
        chromaticity = (1.0, 1.0),
        crab_frequency = 394.0e6,
        crab_strength_x = (tan(12.5e-3) / sqrt(150.0 * 0.55), 0.0, 0.0),
        crab_strength_y = (0.0, 0.0, 0.0),
        crab_phase = (0.0, 0.0, 0.0),
        radiation_damping_turns = (4000.0, 4000.0, 2000.0),
    ),

    proton = (
        charge = 1.0,
        mass = PMASS_EV,
        energy = 275.0e9,
        n_particle = 0.6881e11,
        n_macro = 1024000,
        cutoff = 5.0,
        sigma = (95.0e-6, 8.5e-6, 6.0e-2),
        beta = (0.8, 0.072, 6.0e-2 / 6.6e-4),
        alpha = (0.0, 0.0),
        alpha6 = (0.0, 0.0, 0.0),
        crab_beta = (1300.0, 30.0, 6.0e-2 / 6.6e-4),
        tune = (0.228, 0.210, -0.01),
        chromaticity = (2.0, 2.0),
        crab_frequency = 197.0e6,
        crab_strength_x = (
            tan(12.5e-3) / sqrt(1300.0 * 0.8) * 4.0 / 3.0,
            -tan(12.5e-3) / sqrt(1300.0 * 0.8) / 3.0,
            0.0,
        ),
        crab_strength_y = (0.0, 0.0, 0.0),
        crab_phase = (0.0, 0.0, 0.0),
    ),

    slicing = (
        zslice = 15,
        center = :centroid,
    ),

    solver = (
        pic_grid = (128, 128),
        pic_deposit_method = :CIC,
        pic_green_type = :integrated,
        min_sigma = 1.0e-12,
        luminosity_scale = nothing,
    ),

    output = (
        luminosity_file = "pic_hcc_notebook.lum",
        electron_moment_file = "pic_hcc_notebook.ele.h5",
        proton_moment_file = "pic_hcc_notebook.pro.h5",
        moment_start = 0,
        moment_step = 100,
        moment_capacity = 1024,
    ),
)

set_global_rng!(seed = input.seed, method = :philox)
input.case_name


In [ ]:
ele = input.electron
beam_ele = Beam(n_macro_ele, policy, Float64;
    beta = ele.beta,
    alpha = ele.alpha,
    sigma = ele.sigma,
    cutoff = ele.cutoff,
    rng_id = 1,
    charge = ele.charge,
    mc2 = ele.mass,
    E0 = ele.energy,
    r0 = RE * ME0 / ele.mass,
    npart = ele.n_particle,
)

pro = input.proton
beam_pro = Beam(n_macro_pro, policy, Float64;
    beta = pro.beta,
    alpha = pro.alpha,
    sigma = pro.sigma,
    cutoff = pro.cutoff,
    rng_id = 2,
    charge = pro.charge,
    mc2 = pro.mass,
    E0 = pro.energy,
    r0 = RE * ME0 / pro.mass,
    npart = pro.n_particle,
)

eltype(beam_ele.rep.x) === Float64 || error("electron beam tracking arrays must be Float64")
eltype(beam_pro.rep.x) === Float64 || error("proton beam tracking arrays must be Float64")

(beam_statistics(beam_ele).rms, beam_statistics(beam_pro).rms)

In [ ]:
slicing = LongitudinalSlicing(;
    method = :normal_quantile,
    nslices = input.slicing.zslice,
    center_position = input.slicing.center,
)

pic_luminosity_schedule =
    pic_luminosity_every < 0 ? error("pic_luminosity_every must be >= 0") :
    pic_luminosity_every == 0 ? AtTurns(Int[]) :
    pic_luminosity_every == 1 ? nothing :
    EveryNSteps(step = pic_luminosity_every)

solver = if solver_kind == :gaussian
    GaussianPoissonSolver(;
        slicing = slicing,
        min_sigma = input.solver.min_sigma,
        luminosity_scale = input.solver.luminosity_scale,
    )
elseif solver_kind == :PIC
    PICPoissonSolver(;
        slicing = slicing,
        luminosity_scale = input.solver.luminosity_scale,
        grid = input.solver.pic_grid,
        deposit_method = input.solver.pic_deposit_method,
        green_type = input.solver.pic_green_type,
        green_cache = pic_green_cache,
        longitudinal_kick = pic_longitudinal_kick,
        batch_mode = pic_batch_mode,
        luminosity_schedule = pic_luminosity_schedule,
    )
else
    error("unknown solver_kind=$(solver_kind); use :PIC or :gaussian")
end

solver


In [ ]:
electron_tccb2ip = Linear6DSpec{Float64}(;
    beta1 = ele.crab_beta,
    beta2 = ele.beta,
    alpha1 = ele.alpha6,
    alpha2 = ele.alpha6,
    dmu = (pi / 2.0, 0.0, 0.0),
)
electron_tccb2ip_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(electron_tccb2ip))))

electron_ip2tcca = Linear6DSpec{Float64}(;
    beta1 = ele.beta,
    beta2 = ele.crab_beta,
    alpha1 = ele.alpha6,
    alpha2 = ele.alpha6,
    dmu = (pi / 2.0, 0.0, 0.0),
)
electron_ip2tcca_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(electron_ip2tcca))))

electron_tccb = ThinCrabCavitySpec{3}(ele.crab_frequency;
    strengthX = ele.crab_strength_x,
    strengthY = ele.crab_strength_y,
    phase = ele.crab_phase,
)
electron_tcca = ThinCrabCavitySpec{3}(ele.crab_frequency;
    strengthX = ele.crab_strength_x,
    strengthY = ele.crab_strength_y,
    phase = ele.crab_phase,
)

electron_one_turn = Linear6DSpec{Float64}(;
    beta1 = ele.beta,
    beta2 = ele.beta,
    alpha1 = ele.alpha6,
    alpha2 = ele.alpha6,
    dmu = 2pi .* ele.tune,
)
electron_chrom = ChromaticityKickSpec{Float64}(;
    xi = ele.chromaticity,
    beta = (ele.beta[1], ele.beta[2]),
    alpha = ele.alpha,
)
electron_rad = LumpedRadSpec{Float64}(;
    damping_turns = ele.radiation_damping_turns,
    beta = ele.beta,
    alpha = ele.alpha6,
    sigma = ele.sigma,
    is_damping = true,
    is_excitation = true,
    rng_id = 3,
)

proton_tccb2ip = Linear6DSpec{Float64}(;
    beta1 = pro.crab_beta,
    beta2 = pro.beta,
    alpha1 = pro.alpha6,
    alpha2 = pro.alpha6,
    dmu = (pi / 2.0, 0.0, 0.0),
)
proton_tccb2ip_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(proton_tccb2ip))))

proton_ip2tcca = Linear6DSpec{Float64}(;
    beta1 = pro.beta,
    beta2 = pro.crab_beta,
    alpha1 = pro.alpha6,
    alpha2 = pro.alpha6,
    dmu = (pi / 2.0, 0.0, 0.0),
)
proton_ip2tcca_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(proton_ip2tcca))))

proton_tccb = ThinCrabCavitySpec{3}(pro.crab_frequency;
    strengthX = pro.crab_strength_x,
    strengthY = pro.crab_strength_y,
    phase = pro.crab_phase,
)
proton_tcca = ThinCrabCavitySpec{3}(pro.crab_frequency;
    strengthX = pro.crab_strength_x,
    strengthY = pro.crab_strength_y,
    phase = pro.crab_phase,
)

proton_one_turn = Linear6DSpec{Float64}(;
    beta1 = pro.beta,
    beta2 = pro.beta,
    alpha1 = pro.alpha6,
    alpha2 = pro.alpha6,
    dmu = 2pi .* pro.tune,
)
proton_chrom = ChromaticityKickSpec{Float64}(;
    xi = pro.chromaticity,
    beta = (pro.beta[1], pro.beta[2]),
    alpha = pro.alpha,
)

lb = LorentzBoostSpec(input.crossing_angle)
rlb = RevLorentzBoostSpec(input.crossing_angle)
ip = StrongStrongCollision(:ip; poisson_solver = solver)

"elements ready"

In [ ]:
mkpath(input.result_dir)
luminosity_path = joinpath(input.result_dir, input.output.luminosity_file)
electron_moment_path = joinpath(input.result_dir, input.output.electron_moment_file)
proton_moment_path = joinpath(input.result_dir, input.output.proton_moment_file)
moment_schedule = EveryNSteps(;
    start = input.output.moment_start,
    stop = input.total_turns,
    step = input.output.moment_step,
)

line_ele = (
    electron_tccb2ip_inv,
    electron_tccb,
    electron_tccb2ip,
    lb,
    ip,
    rlb,
    electron_ip2tcca,
    electron_tcca,
    electron_ip2tcca_inv,
    electron_one_turn,
    electron_chrom,
    electron_rad,
    ScheduledObserver(
        MomentObserver(electron_moment_path; capacity = input.output.moment_capacity),
        moment_schedule,
    ),
)

line_pro = (
    proton_tccb2ip_inv,
    proton_tccb,
    proton_tccb2ip,
    lb,
    ip,
    rlb,
    proton_ip2tcca,
    proton_tcca,
    proton_ip2tcca_inv,
    proton_one_turn,
    proton_chrom,
    ScheduledObserver(
        MomentObserver(proton_moment_path; capacity = input.output.moment_capacity),
        moment_schedule,
    ),
)

task = StrongStrongTask(line_ele, line_pro;
    luminosity_path = luminosity_path,
)

(length(line_ele), length(line_pro), luminosity_path, electron_moment_path, proton_moment_path)

In [ ]:
@time execute!(task, beam_ele, beam_pro; turns = turns)

stats_ele = beam_statistics(beam_ele)
stats_pro = beam_statistics(beam_pro)
println("turns = ", turns)
println("n_macro_ele = ", n_macro_ele)
println("n_macro_pro = ", n_macro_pro)
println("use_gpu = ", use_gpu)
println("solver_kind = ", solver_kind)
if solver_kind == :PIC
    println("pic_longitudinal_kick = ", pic_longitudinal_kick)
    println("pic_batch_mode = ", pic_batch_mode)
    println("pic_green_cache = ", pic_green_cache)
    println("pic_luminosity_every = ", pic_luminosity_every)
    println("cuda_pic_indexed_wavefront = ", get(ENV, "OCTOPUS_CUDA_PIC_INDEXED_WAVEFRONT", "0"))
end
println("luminosity = ", luminosity_path)
println("electron moments = ", electron_moment_path)
println("proton moments = ", proton_moment_path)
println("electron rms = ", stats_ele.rms)
println("proton rms = ", stats_pro.rms)


In [ ]:
electron_output = MomentOutputFile(electron_moment_path)
proton_output = MomentOutputFile(proton_moment_path)

println("electron columns = ", column_names(electron_output))
println("proton columns = ", column_names(proton_output))
println("electron data size = ", size(read(electron_output)))
println("proton data size = ", size(read(proton_output)))